In [1]:
import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,precision_recall_fscore_support)

# Constants

In [19]:
SEED = 42
IMG = 224
BATCH = 32
CNN_EPOCHS = 30
TL_HEAD_EPOCHS = 5
TL_FINE_EPOCHS = 12
DATASET = "BrandonFors/Plant-Diseases-PlantVillage-Dataset"
RES = "results"
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


In [3]:
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
for sub in ["cnn", "transfer", "gradcam", "models"]:
    os.makedirs(f"{RES}/{sub}", exist_ok=True)

In [4]:
raw = load_dataset(DATASET)
data = concatenate_datasets(list(raw.values()))          # merge splits,do my own split
img_col = "image"
lbl_col = "label"

README.md:   0%|          | 0.00/6.64k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  321MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  362MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  170MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43456 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10849 [00:00<?, ? examples/s]

In [5]:
class_names = data.features[lbl_col].names
num_classes = len(class_names)
labels = np.array(data[lbl_col])
print(f"{len(data)} images, {num_classes} classes")

54305 images, 38 classes


In [6]:
idx_all = np.arange(len(data))
idx_train, idx_tmp = train_test_split(idx_all, test_size=0.30, stratify=labels, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=labels[idx_tmp], random_state=SEED)
print(f"train as (70%) {len(idx_train)} | val as (15%) {len(idx_val)} | test as (15%) {len(idx_test)}")

train as (70%) 38013 | val as (15%) 8146 | test as (15%) 8146


In [7]:
def load_image(i):
    img = data[int(i)][img_col].convert("RGB").resize((IMG, IMG))
    return np.asarray(img, dtype=np.float32)

In [8]:
def make_ds(indices, training):
    def gen():
        order = np.random.permutation(indices) if training else indices
        for i in order:
            yield load_image(i), labels[i]
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(tf.TensorSpec((IMG, IMG, 3), tf.float32), tf.TensorSpec((), tf.int64)))
    ds = ds.batch(BATCH)
    # from_generator has unknown length -> progress bar shows "15/Unknown". Tell tf.data the real number of batches.
    n_batches = -(-len(indices) // BATCH)          # ceil division
    ds = ds.apply(tf.data.experimental.assert_cardinality(n_batches))
    return ds.prefetch(tf.data.AUTOTUNE)

In [9]:
train_ds = make_ds(idx_train, True)
val_ds = make_ds(idx_val, False)
test_ds = make_ds(idx_test, False)
y_test = labels[idx_test]

cw = compute_class_weight("balanced", classes=np.arange(num_classes), y=labels[idx_train])
class_weight = {i: float(w) for i, w in enumerate(cw)}

In [10]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augment")

In [11]:
def get_callbacks(name):
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2,
                                          min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(RES, "models", f"best_{name}.keras"),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=1),
        keras.callbacks.CSVLogger(f"{RES}/{name}/training_log_{np.random.randint(1e6)}.csv"),
    ]

In [12]:
def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=["accuracy"])

In [13]:
def plot_history(histories, name, boundary=None):
    acc, val_acc, loss, val_loss = [], [], [], []

    for h in histories:
        acc += h.history["accuracy"]
        val_acc += h.history["val_accuracy"]
        loss += h.history["loss"]
        val_loss += h.history["val_loss"]

    pd.DataFrame({"accuracy": acc, "val_accuracy": val_acc,
                  "loss": loss, "val_loss": val_loss}).to_csv(f"{RES}/{name}/history.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

    for a, tr, va, title in [(ax[0], acc, val_acc, "Accuracy"), (ax[1], loss, val_loss, "Loss")]:
        a.plot(tr, label="train")
        a.plot(va, label="validation")
        if boundary:
            a.axvline(boundary - 0.5, color="gray", ls="--", label="fine-tuning starts")
        a.set_title(f"{name} - {title}"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)

    plt.tight_layout();
    plt.savefig(f"{RES}/{name}/training_curves.png", dpi=150)
    plt.close()

In [14]:
def evaluate(model, name):
    logits = model.predict(test_ds, verbose=1)
    y_pred = logits.argmax(1)

    acc = accuracy_score(y_test, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
    top5 = float(np.mean([y in np.argsort(l)[-5:] for y, l in zip(y_test, logits)]))
    summary = {"model": name, "accuracy": acc, "top5_accuracy": top5,
               "precision_macro": p, "recall_macro": r, "f1_macro": f1,
               "precision_weighted": pw, "recall_weighted": rw, "f1_weighted": fw}
    json.dump(summary, open(f"{RES}/{name}/metrics.json", "w"), indent=2)
    print(json.dumps(summary, indent=2))

    report = classification_report(y_test, y_pred, target_names=class_names,
                                  output_dict=True, zero_division=0)
    df = pd.DataFrame(report).T
    df.to_csv(f"{RES}/{name}/classification_report.csv")
    open(f"{RES}/{name}/classification_report.txt", "w").write(
        classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    for mat, fname, fmt in [(cm, "confusion_matrix.png", "d"), (cm_norm, "confusion_matrix_normalized.png", ".2f")]:
        plt.figure(figsize=(20, 17))
        sns.heatmap(mat, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{name} - Confusion matrix")
        plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
        plt.tight_layout(); plt.savefig(f"{RES}/{name}/{fname}", dpi=130); plt.close()

    f1s = df.loc[class_names, "f1-score"].sort_values()
    plt.figure(figsize=(9, 11))
    plt.barh(f1s.index, f1s.values, color="seagreen")
    plt.xlim(max(0, f1s.min() - 0.05), 1.0); plt.xlabel("F1-score")
    plt.title(f"{name} - per-class F1"); plt.tight_layout()
    plt.savefig(f"{RES}/{name}/per_class_f1.png", dpi=150); plt.close()
    return summary, y_pred

In [15]:
def conv_block(x, filters, pool=True):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    if pool:
        x = layers.MaxPooling2D()(x)
    return layers.Dropout(0.25)(x)


def build_cnn():
    f_in = keras.Input((IMG, IMG, 3))
    x = conv_block(f_in, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256, pool=False)            # last conv map (28x28) is used by Grad-CAM
    features = keras.Model(f_in, x, name="features")

    inp = keras.Input((IMG, IMG, 3))
    x = augment(inp)
    x = layers.Rescaling(1 / 255.0)(x)
    x = features(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, name="logits")(x)
    return keras.Model(inp, out, name="cnn")

In [16]:
cnn = build_cnn()
cnn.summary()
compile_model(cnn, 1e-3)

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augment (Sequential)            │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ features (Functional)           │ (None, 28, 28, 256)    │     1,175,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,250,694 (4.77 MB)

 Trainable params: 1,248,774 (4.76 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [17]:
h = cnn.fit(train_ds, validation_data=val_ds, epochs=CNN_EPOCHS,
            class_weight=class_weight, callbacks=get_callbacks("cnn"), verbose=1)

Epoch 1/30
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.2884 - loss: 2.5101
Epoch 1: val_loss improved from None to 1.46849, saving model to results/models/best_cnn.keras

Epoch 1: finished saving model to results/models/best_cnn.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 459s 373ms/step - accuracy: 0.3931 - loss: 2.0709 - val_accuracy: 0.5522 - val_loss: 1.4685 - learning_rate: 0.0010
Epoch 2/30
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.5584 - loss: 1.4312
Epoch 2: val_loss improved from 1.46849 to 1.27086, saving model to results/models/best_cnn.keras

Epoch 2: finished saving model to results/models/best_cnn.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 441s 371ms/step - accuracy: 0.5900 - loss: 1.3205 - val_accuracy: 0.5927 - val_loss: 1.2709 - learning_rate: 0.0010
Epoch 3/30
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.6685 - loss: 1.0293
Epoch 3: val_loss did not improve from 1.27086
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 434s 366ms/step - accuracy: 0.6883 

In [30]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [18]:
plot_history([h], "cnn")
cnn.save(f"{RES}/models/cnn.keras")
cnn_metrics, cnn_pred = evaluate(cnn, "cnn")

255/255 ━━━━━━━━━━━━━━━━━━━━ 36s 140ms/step
{
  "model": "cnn",
  "accuracy": 0.9129634176282838,
  "top5_accuracy": 0.9970537687208446,
  "precision_macro": 0.8762094690733531,
  "recall_macro": 0.9249214868406511,
  "f1_macro": 0.8922148342183892,
  "precision_weighted": 0.9264988910590594,
  "recall_weighted": 0.9129634176282838,
  "f1_weighted": 0.9150997694268617
}


In [20]:
def build_transfer():
    base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                             input_shape=(IMG, IMG, 3))
    # EfficientNet has its own preprocessing (expects 0-255 input); wrap it so Grad-CAM finds "features"
    base = keras.Model(base.input, base.output, name="features")
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3))
    x = augment(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, name="logits")(x)
    return keras.Model(inp, out, name="transfer"), base


In [21]:
tl, base = build_transfer()
compile_model(tl, 1e-3)
h1 = tl.fit(train_ds, validation_data=val_ds, epochs=TL_HEAD_EPOCHS,
            class_weight=class_weight, callbacks=get_callbacks("transfer"))

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.6869 - loss: 1.5057
Epoch 1: val_loss improved from None to 0.34325, saving model to results/models/best_transfer.keras

Epoch 1: finished saving model to results/models/best_transfer.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 238s 187ms/step - accuracy: 0.8111 - loss: 0.8871 - val_accuracy: 0.9105 - val_loss: 0.3432 - learning_rate: 0.0010
Epoch 2/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.9015 - loss: 0.3827
Epoch 2: val_loss improved from 0.34325 to 0.23935, saving model to results/models/best_transfer.keras

Epoch 2: finished saving model to results/models/best_transfer.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 190s 160ms/step - accuracy: 0.9060 - loss: 0.3565 - val_accuracy: 0.9326 - val_loss: 0.2393 - learning_rate: 0.0010
Epoch 3/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 0.9156 - loss: 0.2993
Epoch 3: val_loss did not improve from 0.2393

In [22]:
# fine-tune: unfreeze the top of the backbone, keep BatchNorm frozen
base.trainable = True
for l in base.layers[:-60]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
compile_model(tl, 1e-4)
h2 = tl.fit(train_ds, validation_data=val_ds, epochs=TL_FINE_EPOCHS,
            class_weight=class_weight, callbacks=get_callbacks("transfer"))
plot_history([h1, h2], "transfer", boundary=len(h1.history["loss"]))
tl.save(f"{RES}/models/transfer_efficientnetb0.keras")
tl_metrics, tl_pred = evaluate(tl, "transfer")

Epoch 1/12
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.9454 - loss: 0.1656
Epoch 1: val_loss improved from None to 0.09565, saving model to results/models/best_transfer.keras

Epoch 1: finished saving model to results/models/best_transfer.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 233s 186ms/step - accuracy: 0.9558 - loss: 0.1359 - val_accuracy: 0.9683 - val_loss: 0.0957 - learning_rate: 1.0000e-04
Epoch 2/12
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.9697 - loss: 0.0910
Epoch 2: val_loss improved from 0.09565 to 0.07350, saving model to results/models/best_transfer.keras

Epoch 2: finished saving model to results/models/best_transfer.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 219s 185ms/step - accuracy: 0.9722 - loss: 0.0852 - val_accuracy: 0.9753 - val_loss: 0.0735 - learning_rate: 1.0000e-04
Epoch 3/12
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.9779 - loss: 0.0696
Epoch 3: val_loss did not improve from 0.07350
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 213s 17

In [23]:
comp = pd.DataFrame([cnn_metrics, tl_metrics]).set_index("model")
comp.to_csv(f"{RES}/model_comparison.csv")
comp[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].T.plot(
    kind="bar", figsize=(9, 5), ylim=(comp[["accuracy", "f1_macro"]].values.min() - 0.05, 1.0))
plt.title("CNN vs Transfer learning (test set)"); plt.xticks(rotation=0); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RES}/model_comparison.png", dpi=150); plt.close()
print(comp.round(4))


          accuracy  top5_accuracy  precision_macro  recall_macro  f1_macro  \
model                                                                        
cnn         0.9130         0.9971           0.8762        0.9249    0.8922   
transfer    0.9935         1.0000           0.9932        0.9899    0.9914   

          precision_weighted  recall_weighted  f1_weighted  
model                                                       
cnn                   0.9265           0.9130       0.9151  
transfer              0.9936           0.9935       0.9935  


In [24]:
def grad_cam(model, img, class_idx=None):
    """img: float32 (IMG, IMG, 3) in 0-255. Uses the 'features' block + the head after it."""
    feat = model.get_layer("features")
    head = model.layers[model.layers.index(feat) + 1:]
    x = tf.convert_to_tensor(img[None])
    # the Rescaling layer of the CNN sits before `features`, so apply the pre-feature layers too
    pre = model.layers[1:model.layers.index(feat)]          # skip InputLayer
    pre = [l for l in pre if not isinstance(l, keras.Sequential)]   # skip augmentation
    for l in pre:
        x = l(x)
    with tf.GradientTape() as tape:
        fmap = feat(x, training=False)
        tape.watch(fmap)
        out = fmap
        for l in head:
            out = l(out, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(out[0]))
        score = out[:, class_idx]
    grads = tape.gradient(score, fmap)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * fmap, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG, IMG)).numpy().squeeze()
    probs = tf.nn.softmax(out[0]).numpy()
    return cam, class_idx, float(probs[class_idx])


def overlay(img, cam, alpha=0.45):
    heat = plt.cm.jet(cam)[..., :3]
    return np.clip((1 - alpha) * (img / 255.0) + alpha * heat, 0, 1)


def gradcam_figure(model, name, indices, fname, title):
    n = len(indices)
    fig, ax = plt.subplots(n, 3, figsize=(10, 3.3 * n))
    ax = np.atleast_2d(ax)
    for row, i in enumerate(indices):
        img = load_image(i)
        cam, pred, conf = grad_cam(model, img)
        true = labels[i]
        ok = pred == true
        ax[row, 0].imshow(img / 255.0); ax[row, 0].set_title(f"True: {class_names[true]}", fontsize=8)
        ax[row, 1].imshow(cam, cmap="jet"); ax[row, 1].set_title("Grad-CAM heatmap", fontsize=8)
        ax[row, 2].imshow(overlay(img, cam))
        ax[row, 2].set_title(f"Pred: {class_names[pred]} ({conf:.2f})", fontsize=8,
                             color="green" if ok else "red")
        for a in ax[row]:
            a.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{RES}/gradcam/{fname}", dpi=140); plt.close()



rng = np.random.RandomState(SEED)
sample_pos = rng.choice(len(idx_test), 8, replace=False)
sample_idx = idx_test[sample_pos]

for model, name, pred in [(cnn, "cnn", cnn_pred), (tl, "transfer", tl_pred)]:
    gradcam_figure(model, name, sample_idx[:4], f"{name}_gradcam_samples_1.png", f"{name}: Grad-CAM on test samples")
    gradcam_figure(model, name, sample_idx[4:], f"{name}_gradcam_samples_2.png", f"{name}: Grad-CAM on test samples")
    wrong = idx_test[np.where(pred != y_test)[0]]
    if len(wrong):
        gradcam_figure(model, name, wrong[:6], f"{name}_gradcam_misclassified.png",
                       f"{name}: Grad-CAM on misclassified images")



In [25]:
fig, ax = plt.subplots(4, 3, figsize=(10, 13))
for row, i in enumerate(sample_idx[:4]):
    img = load_image(i)
    cam_c, p_c, _ = grad_cam(cnn, img)
    cam_t, p_t, _ = grad_cam(tl, img)
    ax[row, 0].imshow(img / 255.0); ax[row, 0].set_title(f"True: {class_names[labels[i]]}", fontsize=8)
    ax[row, 1].imshow(overlay(img, cam_c)); ax[row, 1].set_title(f"CNN: {class_names[p_c]}", fontsize=8)
    ax[row, 2].imshow(overlay(img, cam_t)); ax[row, 2].set_title(f"Transfer: {class_names[p_t]}", fontsize=8)
    for a in ax[row]:
        a.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/gradcam/cnn_vs_transfer_gradcam.png", dpi=140); plt.close()
